# Naive Bayes - Stratified 5-Fold CV + Holdout Test
- Grid search on 80% train using StratifiedGroupKFold with your strat_key
- Best config selected by mean weighted F1 across 5 folds
- All 5 metrics (Accuracy, Precision, Recall, F1, AUROC) reported per fold + mean ± std
- Final evaluation on fixed 20% holdout test

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings
from dotenv import load_dotenv
warnings.filterwarnings('ignore')

from sklearn.naive_bayes import ComplementNB, GaussianNB
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, classification_report
)
import wandb
import os
load_dotenv()

True

In [ ]:
wandb_key = os.environ.get("WANDB_API_KEY")
if not wandb_key:
    raise ValueError("Set WANDB_API_KEY environment variable")
wandb.login(key=wandb_key)

wandb.init(
    project = "commitment-mining",
    name = "ml-naive-bayes-LaBSE-aug",
    config = {
        "model": "navie-bayes",
    },
    tags=["naive-bayes", "machine-learning", "LaBSE", "augmented"],
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/rupak/.netrc
wandb: Currently logged in as: nk23041720 (nk23041720-kathmandu-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 2. Load Data

In [ ]:
TRAIN_PATH  = 'Commitment-Mining/dataset/train_80p-aug.xlsx'
TEST_PATH   = 'Commitment-Mining/dataset/test_20p.xlsx'

TEXT_COL    = 'statements (ne)'
LABEL_COL   = 'final_label'
RANDOM_SEED = 42

train_df = pd.read_excel(TRAIN_PATH)
test_df  = pd.read_excel(TEST_PATH)

for df in [train_df, test_df]:
    df[TEXT_COL]  = df[TEXT_COL].astype(str).str.strip()
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

wandb.log({
    "train_size" : len(train_df),
    "test_size"  : len(test_df),
})

label_counts = train_df[LABEL_COL].value_counts().to_dict()
wandb.log(label_counts)

## 3. Build Stratification Key

In [5]:
# Bin sentence length into buckets
train_df['sentence_length'] = train_df[TEXT_COL].apply(
    lambda x: pd.cut(
        [len(x.split())],
        bins=[0, 5, 10, 20, 50, 999],
        labels=['xs', 's', 'm', 'l', 'xl']
    )[0]
)

# Build strat key exactly as specified
train_df['strat_key'] = (
    train_df['province'].astype(str)           + '_' +
    train_df['sentence_length'].astype(str)    + '_' +
    train_df['district/gaupalika'].astype(str) + '_' +
    train_df[LABEL_COL].astype(str)
)

# Merge rare keys (< 5 members) into 'rare' to avoid CV split issues
counts = train_df['strat_key'].value_counts()
rare   = counts[counts < 5].index
train_df['strat_key'] = train_df['strat_key'].apply(
    lambda x: 'rare' if x in rare else x
)


## 4. Prepare Features & Labels

In [6]:
le = LabelEncoder()

X_train = train_df[TEXT_COL].to_numpy(dtype=str)
y_train = le.fit_transform(train_df[LABEL_COL].values)
groups  = train_df['strat_key'].values   # used by StratifiedGroupKFold

X_test  = test_df[TEXT_COL].to_numpy(dtype=str)
y_test  = le.transform(test_df[LABEL_COL].values)

labse = SentenceTransformer('sentence-transformers/LaBSE')


X_train_emb = labse.encode(X_train.tolist(), batch_size=32, show_progress_bar=True, convert_to_numpy=True  )
X_test_emb  = labse.encode(X_test.tolist(),  batch_size=32, show_progress_bar=True, convert_to_numpy=True  )

wandb.log({
    "num_classes": len(le.classes_),
    "classes": ", ".join(map(str, le.classes_)),
    "x_train_samples": X_train.shape[0],
    "x_train_features": X_train.shape[1] if len(X_train.shape) > 1 else 1,
    "x_test_samples": X_test.shape[0],
    "x_test_features": X_test.shape[1] if len(X_test.shape) > 1 else 1,
})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/61 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

## 5. Grid Search - Find Best Config
- Uses `StratifiedGroupKFold` so your strat_key is respected
- Best config = highest mean weighted F1 across 5 validation folds
- `refit=True` → best config is refit on full 80% train automatically

In [ ]:
pipeline = GaussianNB()

param_grid = {
    'var_smoothing': [1e-9, 1e-7, 1e-5, 1e-3]
}
# Total: 4 configs x 5 folds = 20 fits


# StratifiedGroupKFold: respects both class balance AND your strat_key groups
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

grid_search = GridSearchCV(
    estimator  = pipeline,
    param_grid = param_grid,
    cv         = sgkf,
    scoring    = 'f1_macro',   # selects best config by weighted F1
    n_jobs     = -1,
    verbose    = 1,
    refit      = True
)

# groups= is now correctly used by StratifiedGroupKFold
grid_search.fit(X_train_emb, y_train, groups=groups)

print('\n✓ Grid search complete.') 
print(f'Best CV weighted F1 : {grid_search.best_score_:.4f}') 
print(f'Best params:') 
for k, v in grid_search.best_params_.items(): 
    print(f' {k}: {v}')

wandb.config.update({
    "best_cv_weighted_f1": grid_search.best_score_,
    **{f"best_param/{k}": v for k, v in grid_search.best_params_.items()}
})

Fitting 5 folds for each of 144 candidates, totalling 720 fits



✓ Grid search complete.
Best CV weighted F1 : 0.7386
Best params:
 class_weight: None
 criterion: gini
 max_depth: 5
 min_samples_leaf: 1
 min_samples_split: 10


## 6. Per-Fold CV - All 5 Metrics with Best Config
GridSearchCV only tracked F1 during search.  
Now rerun best config across 5 folds to get Accuracy, Precision, Recall, F1, AUROC per fold.

In [ ]:
def compute_metrics(y_true, y_pred, y_proba):
    """Compute all 5 metrics. Returns a dict."""
    return {
        'accuracy'  : accuracy_score(y_true, y_pred),
        'precision' : precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall'    : recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1'        : f1_score(y_true, y_pred, average='macro', zero_division=0),
        'auroc'     : roc_auc_score(y_true, y_proba)
    }

fold_metrics = []

for fold, (tr_idx, val_idx) in enumerate(
    sgkf.split(X_train, y_train, groups=groups)
):
    X_tr, X_val = X_train_emb[tr_idx], X_train_emb[val_idx]
    y_tr,  y_val  = y_train[tr_idx],  y_train[val_idx]

    # Fresh clone of best config — same hyperparameters, retrained from scratch
    fold_model = clone(grid_search.best_estimator_)
    fold_model.fit(X_tr, y_tr)

    y_pred  = fold_model.predict(X_val)
    y_proba = fold_model.predict_proba(X_val)[:, 1]

    m = compute_metrics(y_val, y_pred, y_proba)
    m['fold'] = fold + 1
    fold_metrics.append(m)

    print(f"Fold {fold+1} | "
          f"Acc: {m['accuracy']:.4f} | "
          f"Prec: {m['precision']:.4f} | "
          f"Rec: {m['recall']:.4f} | "
          f"F1: {m['f1']:.4f} | "
          f"AUROC: {m['auroc']:.4f}")

# Build results dataframe
fold_df = pd.DataFrame(fold_metrics).set_index('fold')
mean_row = fold_df.mean().rename('mean')
std_row  = fold_df.std().rename('std')
cv_summary = pd.concat([fold_df, mean_row.to_frame().T, std_row.to_frame().T])

print('\n=== CV Results (best config) ===') 
print(cv_summary.round(4))

wandb.log({
    "cv_results": wandb.Table(dataframe=cv_summary.round(4))
})

Fold 1 | Acc: 0.7264 | Prec: 0.7418 | Rec: 0.7264 | F1: 0.7244 | AUROC: 0.7896
Fold 2 | Acc: 0.8179 | Prec: 0.8190 | Rec: 0.8179 | F1: 0.8175 | AUROC: 0.8459
Fold 3 | Acc: 0.6419 | Prec: 0.6398 | Rec: 0.6419 | F1: 0.6351 | AUROC: 0.6726
Fold 4 | Acc: 0.8076 | Prec: 0.8185 | Rec: 0.8076 | F1: 0.8033 | AUROC: 0.7944
Fold 5 | Acc: 0.7095 | Prec: 0.7244 | Rec: 0.7095 | F1: 0.7129 | AUROC: 0.7201

=== CV Results (best config) ===
      accuracy  precision  recall      f1   auroc
1       0.7264     0.7418  0.7264  0.7244  0.7896
2       0.8179     0.8190  0.8179  0.8175  0.8459
3       0.6419     0.6398  0.6419  0.6351  0.6726
4       0.8076     0.8185  0.8076  0.8033  0.7944
5       0.7095     0.7244  0.7095  0.7129  0.7201
mean    0.7407     0.7487  0.7407  0.7386  0.7645
std     0.0731     0.0747  0.0731  0.0742  0.0681


## 7. Final Evaluation on Holdout Test Set (20%)
Best config already refit on full 80% train by GridSearchCV (`refit=True`).  
Evaluated exactly once — never used before this step.

In [9]:
y_pred_test  = grid_search.best_estimator_.predict(X_test_emb)
y_proba_test = grid_search.best_estimator_.predict_proba(X_test_emb)[:, 1]

test_metrics = compute_metrics(y_test, y_pred_test, y_proba_test)

print('=== HOLDOUT TEST SET RESULTS ===') 
print(f" Accuracy : {test_metrics['accuracy']:.4f}") 
print(f" Precision : {test_metrics['precision']:.4f}") 
print(f" Recall : {test_metrics['recall']:.4f}") 
print(f" F1 : {test_metrics['f1']:.4f}") 
print(f" AUROC : {test_metrics['auroc']:.4f}") 
print('\nClassification Report:') 
print(classification_report(y_test, y_pred_test, target_names=le.classes_))


wandb.log({
    "test/accuracy": test_metrics["accuracy"],
    "test/precision": test_metrics["precision"],
    "test/recall": test_metrics["recall"],
    "test/f1": test_metrics["f1"],
    "test/auroc": test_metrics["auroc"],
})

report = classification_report(
    y_test,
    y_pred_test,
    target_names=le.classes_,
    output_dict=True
)

report_df = pd.DataFrame(report).transpose().round(4)

wandb.log({
    "classification_report": wandb.Table(dataframe=report_df)
})

=== HOLDOUT TEST SET RESULTS ===
 Accuracy : 0.7860
 Precision : 0.7857
 Recall : 0.7860
 F1 : 0.7850
 AUROC : 0.8128

Classification Report:
              precision    recall  f1-score   support

           C       0.79      0.84      0.81       270
          NC       0.78      0.72      0.75       216

    accuracy                           0.79       486
   macro avg       0.79      0.78      0.78       486
weighted avg       0.79      0.79      0.78       486



## 8. Final Summary Table
CV mean ± std (validation) vs holdout test - everything in one place for the paper.

In [ ]:
metrics_order = ['accuracy', 'precision', 'recall', 'f1', 'auroc']

summary = pd.DataFrame({
    'CV Mean' : fold_df[metrics_order].mean().round(4),
    'CV Std'  : fold_df[metrics_order].std().round(4),
    'Test'    : pd.Series(test_metrics)[metrics_order].round(4)
})

print('=== PAPER TABLE — Random Forest (Labse) ===') 
print(summary) 

print('\nBest hyperparameters:') 
for k, v in grid_search.best_params_.items(): print(f' {k}: {v}')

wandb.log({
    "paper_table": wandb.Table(dataframe=summary)
})

# Log the best hyperparameters
wandb.config.update({
    **{f"best_param/{k}": v for k, v in grid_search.best_params_.items()}
})

=== PAPER TABLE — Logistic Regression (Labse) ===
           CV Mean  CV Std    Test
accuracy    0.7407  0.0731  0.7860
precision   0.7487  0.0747  0.7857
recall      0.7407  0.0731  0.7860
f1          0.7386  0.0742  0.7850
auroc       0.7645  0.0681  0.8128

Best hyperparameters:
 class_weight: None
 criterion: gini
 max_depth: 5
 min_samples_leaf: 1
 min_samples_split: 10


In [ ]:
wandb.finish()

C,▁
NC,▁
num_classes,▁
test/accuracy,▁
test/auroc,▁
test/f1,▁
test/precision,▁
test/recall,▁
test_size,▁
train_size,▁
+4,...


: 